In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 01, First steps

Companion notebook to [01_first_steps.md](01_first_steps.md). Building a symbolic expression in `jacopy`, declaring properties, and simplifying, the basics.

## Symbols and basic construction

`Symbol(name)` builds an atomic expression; `Integer(n)` is a literal constant; `+`, `*`, `-` produce `Sum`, `Product`, `Neg` under the hood.

In [1]:
from jacopy.core.expr import Symbol, Integer, Sum, Product, Neg

x, y, z = Symbol("x"), Symbol("y"), Symbol("z")

expr = x + y - z
assert expr == Sum(x, y, Neg(z))
print(expr)

(x + y + (-z))


In [2]:
mult = 2 * x
assert mult == Product(Integer(2), x)
print(mult)

(2 * x)


## Declaring properties (`PropertyRegistry`)

Properties, degree, scalarity, graded antisymmetry, …, are declared **externally** on symbols. That's what lets the same symbol be reused in different contexts (different degrees, different algebras).

In [3]:
from jacopy.core.properties import Graded, Scalar
from jacopy.core.registry import PropertyRegistry

reg = PropertyRegistry()
reg.declare(x, Scalar())
reg.declare(y, Scalar())
reg.declare(z, Graded(degree=1))  # z behaves like a 1-form

### Role-driven shortcuts

For common patterns, functions, vector fields, forms, bivectors, `jacopy.library.declarations` exposes `Functions`, `VectorFields`, `Forms`, `Bivector` helpers. Each collapses `Symbol(...)` plus the matching `reg.declare(...)` into a single call.

In [4]:
from jacopy import Functions, VectorFields, Forms, Bivector

reg2 = PropertyRegistry()
f, g = Functions("f g", registry=reg2)
X, Y = VectorFields("X Y", registry=reg2)
alpha, beta = Forms("α β", degree=1, registry=reg2)
pi = Bivector("π", registry=reg2)
print(f, g, X, Y, alpha, beta, pi)

f g X Y α β π


## `simplify`, canonical form

The `simplify(expr, registry)` pipeline: flatten → canonicalize → distribute → flatten → sort_product → collect_terms. With a registry the registered properties (commutativity, degree) are consulted.

In [5]:
from jacopy.algorithms.simplify import simplify

assert simplify(x + x - x) == x
assert simplify(Product(Integer(2), x, Integer(3)), reg) \
    == Product(Integer(6), x)
# Two scalars sort alphabetically:
assert simplify(Product(y, x), reg) == Product(x, y)
print('simplify checks passed')

simplify checks passed


## Display

The `display` layer gives `to_ascii`, `to_latex`, and (if `rich` is installed) a coloured terminal tree.

In [6]:
from jacopy.display import to_ascii, to_latex

e = x + y - z
print('ascii:', to_ascii(e))
print('latex:', to_latex(e))

ascii: x + y - z
latex: x + y - z


## Next step

Layering a bracket on top → Jacobi proof: [02_jacobi_identity.md](02_jacobi_identity.md).